# Legal-RAG: Kaggle + Qdrant + Ollama Cloud

Notebook này đã được tùy chỉnh cấu hình để chạy trên Kaggle:
1. LLM dùng **Ollama Cloud** (model `qwen3.5:cloud` hoặc Groq) thay vì Ollama local daemon.
2. Qdrant hỗ trợ **local path** (`/kaggle/working/local_qdrant_data`) để lưu dữ liệu.
3. Vẫn có thể fallback sang **Qdrant Cloud** nếu cung cấp `QDRANT_URL` + `QDRANT_API_KEY`.

In [1]:
# @title CẤU HÌNH (KAGGLE-READY: OLLAMA CLOUD + QDRANT LOCAL/CLOUD)
import os
from pathlib import Path
import shutil

# Detect Kaggle or Colab runtime
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Qdrant settings
COLLECTION_NAME = "legal_rag_docs_5000" # @param {type:"string"}
DOC_LIMIT =  5000 # @param {type:"integer"}  # 0 = tat ca documents, >0 = gioi han so luong

if IN_KAGGLE:
    QDRANT_LOCAL_PATH = "/kaggle/working/local_qdrant_data"
elif IN_COLAB:
    QDRANT_LOCAL_PATH = "/content/local_qdrant_data"
else:
    QDRANT_LOCAL_PATH = "./local_qdrant_data"

QDRANT_LOCAL_URL = "http://localhost:6333" # @param {type:"string"}
QDRANT_MODE = "cloud" # @param ["local_path", "local_url", "cloud"]

# Cloud Qdrant fallback
QDRANT_URL = "https://bc0bb490-f62b-46b6-9863-5e413732dd93.us-west-1-0.aws.cloud.qdrant.io:6333" # @param {type:"string"}
QDRANT_API_KEY = "<YOUR_QDRANT_API_KEY>" # @param {type:"string"}

# LLM settings
LLM_PROVIDER = "groq" # @param ["ollama", "groq", "gemini", "openai"]
LLM_MODEL = "llama-3.1-8b-instant" # @param {type:"string"}
LLM_API_KEY = "<YOUR_GROQ_API_KEY>" # @param {type:"string"}
HF_API_KEY = "<YOUR_HF_TOKEN>" # @param {type:"string"}

# IMPORTANT:
# - Colab/Kaggle cannot reliably run local Ollama daemon + `ollama run ...` in the background for long.
# - Use Ollama Cloud/Groq endpoint.
LLM_BASE_URL = "https://api.groq.com/openai/v1" # @param {type:"string"}

# Runtime guard
RUN_HEAVY_DEMO = False # @param {type:"boolean"}

# Ensure local path exists when using local_path mode
if QDRANT_MODE == "local_path":
    Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)

# Qdrant env
os.environ["QDRANT_MODE"] = QDRANT_MODE
os.environ["QDRANT_LOCAL_PATH"] = QDRANT_LOCAL_PATH
os.environ["QDRANT_LOCAL_URL"] = QDRANT_LOCAL_URL
os.environ["QDRANT_URL"] = QDRANT_URL
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY
os.environ["DOC_LIMIT"] = str(DOC_LIMIT)

# LLM env
os.environ["LLM_PROVIDER"] = LLM_PROVIDER
os.environ["NOTEBOOK_LLM_MODEL"] = LLM_MODEL
os.environ["OLLAMA_MODEL"] = LLM_MODEL
os.environ["LLM_BASE_URL"] = LLM_BASE_URL
os.environ["OLLAMA_CLOUD_BASE_URL"] = LLM_BASE_URL
os.environ["OLLAMA_CLOUD_API_KEY"] = LLM_API_KEY

# Keep compatibility for existing provider clients
os.environ["GROQ_API_KEY"] = LLM_API_KEY
os.environ["GEMINI_API_KEY"] = LLM_API_KEY
os.environ["OPENAI_API_KEY"] = LLM_API_KEY

# Hugging Face env
os.environ["HF_TOKEN"] = HF_API_KEY
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_API_KEY

try:
    from huggingface_hub import login
    login(token=HF_API_KEY, add_to_git_credential=False)
    print("Hugging Face login: OK")
except Exception as exc:
    print(f"Hugging Face login warning: {exc}")

print(f"IN_KAGGLE={IN_KAGGLE}")
print(f"IN_COLAB={IN_COLAB}")
print(f"QDRANT_MODE={QDRANT_MODE}")
print(f"DOC_LIMIT={DOC_LIMIT} (0 = all)")
print(f"LLM_PROVIDER={LLM_PROVIDER} | MODEL={LLM_MODEL}")
print(f"LLM_BASE_URL={LLM_BASE_URL}")
print(f"HF authenticated={bool(HF_API_KEY)}")
print("Đã cấu hình xong (Kaggle-ready).")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login: OK
IN_KAGGLE=True
IN_COLAB=True
QDRANT_MODE=cloud
DOC_LIMIT=5000 (0 = all)
LLM_PROVIDER=groq | MODEL=llama-3.1-8b-instant
LLM_BASE_URL=https://api.groq.com/openai/v1
HF authenticated=True
Đã cấu hình xong (Kaggle-ready).


# Legal RAG Hybrid Demo Notebook

Notebook nay tap trung vao:
- chunking van ban phap ly xuong muc diem (point-level)
- hybrid retrieval dense + sparse deu dung BGE-M3
- chay local CPU, dung cau hinh key tu file .env
- tai su dung module trong core/ (config, db, llm, nlp)
- co cell uoc tinh thoi gian pipeline tren CPU

In [2]:
%pip install google-genai -q -U qdrant-client datasets pandas scikit-learn accelerate python-dotenv openai ipywidgets sentence-transformers fastembed pypdf python-docx "FlagEmbedding==1.3.5" "transformers==4.48.3" "huggingface-hub<1.0" "tokenizers<0.22"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 55.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 53.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 245.6 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 555.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.7 MB/s eta 0:00

In [3]:
# qdrant sẽ chạy lên cloud
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://bc0bb490-f62b-46b6-9863-5e413732dd93.us-west-1-0.aws.cloud.qdrant.io:6333",
    api_key="<YOUR_QDRANT_API_KEY>",
)

print(qdrant_client.get_collections())

collections=[]


In [4]:
# Cell 2: local setup (.env), Qdrant mode-aware (kaggle/local/cloud), va BGE-M3 cho ca dense + sparse
import huggingface_hub.constants
# Vá lỗi động: Thêm cờ này vào module nếu nó chưa tồn tại
if not hasattr(huggingface_hub.constants, 'HF_HUB_ENABLE_HF_TRANSFER'):
    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False

from __future__ import annotations

import json
import math
import os
import re
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import display
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DENSE_MODEL_NAME = os.getenv("LEGAL_DENSE_MODEL", "BAAI/bge-m3")

# Mode-aware config for Qdrant
QDRANT_MODE = os.getenv("QDRANT_MODE", "cloud").lower() # Updated to default to cloud
QDRANT_LOCAL_PATH = os.getenv("QDRANT_LOCAL_PATH", "./local_qdrant_data")
QDRANT_LOCAL_URL = os.getenv("QDRANT_LOCAL_URL", "http://localhost:6333")
QDRANT_CLOUD_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

TARGET_COLLECTION = globals().get("COLLECTION_NAME", "legal_rag_docs_5000")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"BGE-M3 model: {DENSE_MODEL_NAME}")
print(f"Target Collection: {TARGET_COLLECTION}")
print(f"Qdrant mode: {QDRANT_MODE}")


def init_qdrant_client() -> QdrantClient:
    """Init Qdrant by mode with cloud fallback and lock-safe local path behavior."""
    local_error = None
    cloud_error = None

    existing_client = globals().get("qdrant_client")
    if existing_client is not None:
        try:
            existing_client.get_collections()
            print("Reusing existing Qdrant client from current kernel session.")
            return existing_client
        except Exception:
            pass

    if QDRANT_MODE == "local_path":
        try:
            Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)
            local_client = QdrantClient(path=QDRANT_LOCAL_PATH)
            local_client.get_collections()
            os.environ["QDRANT_ACTIVE_MODE"] = "local_path"
            print(f"Qdrant connected (LOCAL PATH): {QDRANT_LOCAL_PATH}")
            return local_client
        except Exception as exc:
            local_error = exc
            print(f"Canh bao: khong ket noi duoc local path Qdrant ({QDRANT_LOCAL_PATH}): {exc}")

            # Kaggle local mode can hit lock when previous process still owns the path.
            if "already accessed by another instance" in str(exc).lower():
                fallback_path = f"{QDRANT_LOCAL_PATH}_session_{os.getpid()}"
                try:
                    Path(fallback_path).mkdir(parents=True, exist_ok=True)
                    local_client = QdrantClient(path=fallback_path)
                    local_client.get_collections()
                    os.environ["QDRANT_ACTIVE_MODE"] = "local_path_fallback"
                    os.environ["QDRANT_LOCAL_PATH"] = fallback_path
                    print(f"Qdrant connected (LOCAL PATH FALLBACK): {fallback_path}")
                    return local_client
                except Exception as fallback_exc:
                    local_error = f"{exc}; fallback_error={fallback_exc}"

    if QDRANT_MODE == "local_url":
        try:
            local_client = QdrantClient(url=QDRANT_LOCAL_URL)
            local_client.get_collections()
            os.environ["QDRANT_ACTIVE_MODE"] = "local_url"
            print(f"Qdrant connected (LOCAL URL): {QDRANT_LOCAL_URL}")
            return local_client
        except Exception as exc:
            local_error = exc
            print(f"Canh bao: khong ket noi duoc local Qdrant URL ({QDRANT_LOCAL_URL}): {exc}")

    if QDRANT_CLOUD_URL or QDRANT_MODE == "cloud":
        try:
            cloud_client = QdrantClient(url=QDRANT_CLOUD_URL, api_key=QDRANT_API_KEY or None)
            cloud_client.get_collections()
            os.environ["QDRANT_ACTIVE_MODE"] = "cloud"
            print(f"Qdrant connected (CLOUD): {QDRANT_CLOUD_URL}")
            return cloud_client
        except Exception as exc:
            cloud_error = exc

    raise RuntimeError(
        "Khong the ket noi Qdrant. Chon QDRANT_MODE=local_path/local_url hoac cung cap QDRANT_URL/QDRANT_API_KEY. "
        f"local_error={local_error}; cloud_error={cloud_error}"
    )


qdrant_client = init_qdrant_client()


class LocalBGEHybridEncoder:
    """Use one BGE-M3 model to produce both dense and sparse vectors."""

    def __init__(self, model_name: str = "BAAI/bge-m3", device: str = "cpu"):
        self.model_name = model_name
        self.device = device
        self.model = BGEM3FlagModel(model_name, use_fp16=(device == "cuda"), device=device)

    @staticmethod
    def _to_sparse_vector(weights: Dict[str, float]) -> models.SparseVector:
        if not weights:
            return models.SparseVector(indices=[], values=[])
        pairs = [(int(k), float(v)) for k, v in weights.items() if float(v) != 0.0]
        pairs.sort(key=lambda x: x[0])
        return models.SparseVector(
            indices=[idx for idx, _ in pairs],
            values=[val for _, val in pairs],
        )

    def encode_dense(self, texts: List[str], batch_size: int = 16) -> List[List[float]]:
        """Deprecated, preserved for compatibility backwards."""
        return self.encode_hybrid(texts, batch_size)[0]

    def encode_sparse_documents(self, texts: List[str], batch_size: int = 16) -> List[models.SparseVector]:
        """Deprecated, preserved for compatibility backwards."""
        return self.encode_hybrid(texts, batch_size)[1]

    def encode_query_sparse(self, text: str) -> models.SparseVector:
        return self.encode_hybrid([text], batch_size=1)[1][0]

    def encode_hybrid(self, texts: List[str], batch_size: int = 16) -> Tuple[List[List[float]], List[models.SparseVector]]:
        """
        SPEED OPTIMIZATION:
        Perform both sparse and dense encoding in a SINGLE FORWARD PASS!
        Halves the time required for embedding.
        """
        if isinstance(texts, str):
            texts = [texts]

        # Optimize batch processing speeds in Colab/GPU vs CPU
        cuda_batch_size = 256  if torch.cuda.is_available() else batch_size

        outputs = self.model.encode(
            texts,
            batch_size=cuda_batch_size,
            max_length=1024,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )

        # 1. Process Dense
        dense_vecs = outputs["dense_vecs"]
        dense_list = dense_vecs.tolist() if hasattr(dense_vecs, "tolist") else [list(vec) for vec in dense_vecs]

        # 2. Process Sparse
        lexical_weights = outputs["lexical_weights"]
        sparse_list = [self._to_sparse_vector(weights) for weights in lexical_weights]

        return dense_list, sparse_list

print("Loading BGE-M3 for dense + sparse...")
hybrid_encoder = LocalBGEHybridEncoder(model_name=DENSE_MODEL_NAME, device=DEVICE)

probe_dense = hybrid_encoder.encode_hybrid(["kiem tra kich thuoc vector"], batch_size=1)[0][0]
dense_dim = len(probe_dense)
print(f"Dense embedding dimension: {dense_dim}")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
                on_disk=True,
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        },
    )
    print(f"Created Qdrant collection: {COLLECTION_NAME}")
else:
    print(f"Qdrant collection already exists: {COLLECTION_NAME}")

# Keep old variable names for compatibility with later cells
embedder = hybrid_encoder
sparse_encoder = hybrid_encoder

print("Local BGE-M3 hybrid encoder is ready (dense + sparse).")

2026-04-01 10:25:58.143789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775039158.314383      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775039158.363464      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775039158.755311      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039158.755349      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039158.755352      23 computation_placer.cc:177] computation placer alr

Using device: cuda
BGE-M3 model: BAAI/bge-m3
Target Collection: legal_rag_docs_5000
Qdrant mode: cloud
Reusing existing Qdrant client from current kernel session.
Loading BGE-M3 for dense + sparse...


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

initial target device:   0%|          | 0/2 [00:00<?, ?it/s]2026-04-01 10:26:52.338353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775039212.370709     640 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775039212.380459     640 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775039212.403950     640 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039212.403979     640 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039212.4039

Dense embedding dimension: 1024
Created Qdrant collection: legal_rag_docs_5000
Local BGE-M3 hybrid encoder is ready (dense + sparse).


In [5]:
# Cell 3: Nap toan bo du lieu (hoac gioi han bang DOC_LIMIT) de bao phu linh vuc phap ly
print("Dang load FULL metadata/content tu Hugging Face dataset ...")
ds_metadata_all = load_dataset("nhn309261/vietnamese-legal-documents", "metadata", split="data")
ds_content_all = load_dataset("nhn309261/vietnamese-legal-documents", "content", split="data")

import re
import pandas as pd
from typing import Any, List
import os

def split_sector_list(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, list):
        raw_text = ", ".join(str(x) for x in value if str(x).strip())
    else:
        raw_text = str(value).strip()
    if not raw_text or raw_text.lower() == "nan":
        return []

    # SỬA Ở ĐÂY: Chỉ tách theo dấu phẩy (,) kèm khoảng trắng xung quanh
    parts = re.split(r"\s*,\s*", raw_text)

    return sorted(set(part.strip() for part in parts if part and part.strip()))


# Metadata dataframe cho thong ke va chon doc test
df_all = ds_metadata_all.to_pandas().copy()
df_all["id"] = df_all["id"].astype(str)
df_all["sector_list"] = df_all["legal_sectors"].apply(split_sector_list)

# Lọc đầu vào: Chỉ lấy các văn bản có hệ thống phân cấp chuẩn
valid_legal_types = ["Luật", "Nghị định", "Quyết định", "Thông tư", "Hiến pháp", "Bộ luật", "Pháp lệnh", "Nghị quyết"]
df_all = df_all[df_all["legal_type"].isin(valid_legal_types)].copy()

# Intersection id giua metadata/content de tranh lech ban ghi
content_ids = {str(row["id"]) for row in ds_content_all}
valid_ids_all = set(df_all["id"]).intersection(content_ids)

if len(valid_ids_all) < len(df_all):
    print(f"Canh bao: bo qua {len(df_all) - len(valid_ids_all)} metadata record khong co content tuong ung.")

# Chon all hoac limit theo DOC_LIMIT
DOC_LIMIT = int(os.getenv("DOC_LIMIT", str(globals().get("DOC_LIMIT", 0))) or 0)

# Ưu tiên các văn bản có hiệu lực ở những năm mới nhất khi DOC_LIMIT > 0
df_valid = df_all[df_all["id"].isin(valid_ids_all)].copy()

date_col = "issue_date" if "issue_date" in df_valid.columns else "issued_date" if "issued_date" in df_valid.columns else None
if date_col:
    df_valid['parsed_date'] = pd.to_datetime(df_valid[date_col], format='%d/%m/%Y', errors='coerce')
    df_valid = df_valid.sort_values(by='parsed_date', ascending=False)
    df_valid = df_valid.drop(columns=['parsed_date'])

selected_ids = df_valid["id"].tolist()
if DOC_LIMIT > 0:
    selected_ids = selected_ids[:DOC_LIMIT]

selected_id_set = set(selected_ids)

df_selected = df_valid[df_valid["id"].isin(selected_id_set)].copy().reset_index(drop=True)
ds_metadata_selected = ds_metadata_all.filter(lambda row: str(row["id"]) in selected_id_set)
ds_content_selected = ds_content_all.filter(lambda row: str(row["id"]) in selected_id_set)

metadata_by_id = {str(row["id"]): row for row in ds_metadata_selected}
content_by_id = {str(row["id"]): row for row in ds_content_selected}

# Backward-compatible aliases cho cac cell cu phia duoi
df_500 = df_selected
ds_metadata_500 = ds_metadata_selected
ds_content_500 = ds_content_selected

ds_metadata_10000 = ds_metadata_selected
ds_content_10000 = ds_content_selected

print(f"Tong so van ban hop le (đã lọc legal_type) trong dataset: {len(valid_ids_all)}")
print(f"Tong so van ban duoc chon de index: {len(ds_content_selected)}")
if DOC_LIMIT == 0:
    print("Che do: INDEX TAT CA documents")
else:
    print(f"Che do: INDEX GIOI HAN {DOC_LIMIT} documents (ưu tiên văn bản mới nhất)")

sector_stats = (
    df_selected.explode("sector_list")["sector_list"]
    .dropna()
    .value_counts()
)
HOT_SECTORS = sector_stats.head(30).index.tolist()

print("\n--- Top 15 linh vuc theo tan suat trong tap du lieu duoc chon ---")
display(sector_stats.head(15).to_frame("count"))
print(f"So luong HOT_SECTORS phuc vu filter retrieval: {len(HOT_SECTORS)}")

Dang load FULL metadata/content tu Hugging Face dataset ...


README.md: 0.00B [00:00, ?B/s]

metadata/data-00000-of-00001.parquet:   0%|          | 0.00/81.8M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

content/data-00000-of-00011.parquet:   0%|          | 0.00/424M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/518255 [00:00<?, ? examples/s]

Filter:   0%|          | 0/518255 [00:00<?, ? examples/s]

Tong so van ban hop le (đã lọc legal_type) trong dataset: 314416
Tong so van ban duoc chon de index: 5000
Che do: INDEX GIOI HAN 5000 documents (ưu tiên văn bản mới nhất)

--- Top 15 linh vuc theo tan suat trong tap du lieu duoc chon ---


,count
sector_list,
Bộ máy hành chính,2395
Tài chính nhà nước,863
Văn hóa - Xã hội,555
Tài nguyên - Môi trường,441
Bất động sản,395
Thể thao - Y tế,314
Xây dựng - Đô thị,307
Giáo dục,267
Giao thông - Vận tải,250


So luong HOT_SECTORS phuc vu filter retrieval: 27


In [6]:
# Cell 4: in thu 1 van ban trong tap du lieu da duoc chon de kiem tra
if df_selected.empty:
    raise ValueError("Khong co van ban hop le trong dataset sau khi loc.")

# SỬA Ở ĐÂY: Thay df_all thành df_selected
target_id = str(df_selected.iloc[0]["id"])
metadata_raw = metadata_by_id[target_id]
content_raw = content_by_id[target_id]["content"]

print(f"Target test document id: {target_id}")
print("=" * 120)
print("METADATA")
print("=" * 120)
for key, value in metadata_raw.items():
    if key != "id":
        print(f"{key}: {value}")

print("\n" + "=" * 120)
print("RAW CONTENT")
print("=" * 120)
print(content_raw)

Target test document id: 693036
METADATA
document_number: 115/NQ-HĐBCQG
title: Nghị quyết 115/NQ-HĐBCQG năm 2026 hướng dẫn xử lý trường hợp khuyết người ứng cử đại biểu Hội đồng nhân dân vì lý do bất khả kháng do Hội đồng bầu cử Quốc gia ban hành
url: https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Nghi-quyet-115-NQ-HDBCQG-2026-khuyet-nguoi-ung-cu-dai-bieu-Hoi-dong-nhan-dan-ly-do-bat-kha-khang-693036.aspx
legal_type: Nghị quyết
legal_sectors: Bộ máy hành chính
issuing_authority: Hội đồng bầu cử quốc gia
issuance_date: 29/01/2026
signers: Trần Thanh Mẫn:2140

RAW CONTENT
HỘI ĐỒNG BẦU CỬ QUỐC GIA  | CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc 
Số: 115/NQ-HĐBCQG | Hà Nội, ngày 29 tháng 01 năm 2026

NGHỊ QUYẾT

HƯỚNG DẪN XỬ LÝ TRƯỜNG HỢP KHUYẾT NGƯỜI ỨNG CỬ ĐẠI BIỂU HỘI ĐỒNG NHÂN DÂN VÌ LÝ DO BẤT KHẢ KHÁNG

HỘI ĐỒNG BẦU CỬ QUỐC GIA

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 203/2025/QH15;

Căn cứ

In [7]:
# Cell 5: AdvancedLegalChunker theo Regex cay thu bac (Chuong > Dieu > Khoan) + Breadcrumb + Citation Graph metadata

def compact_whitespace(text: str) -> str:
    return re.sub(r"[ \t]+", " ", str(text or "")).strip()


class AdvancedLegalChunker:
    def __init__(self, appendix_chunk_size: int = 1000, max_chunk_size: int = 1500):
        self.appendix_chunk_size = appendix_chunk_size
        self.max_chunk_size = max_chunk_size

        # Hierarchical tree regex
        self.appendix_pattern = re.compile(r"(?im)^\s*(PHU LUC|PHỤ LỤC|DANH MUC|DANH MỤC)\b.*$")
        self.chapter_pattern = re.compile(r"(?im)^\s*(Chương\s+[IVXLCDM0-9]+)\s*(.*)$")
        self.article_pattern = re.compile(r"(?im)^\s*(Điều\s+\d+[A-Za-z0-9\/\-]*)[\.\:\-]?\s*(.*)$")
        self.clause_pattern = re.compile(
            r"(?im)^\s*(Khoản\s+\d+[\.\:\-]?)\s*(.*)$|^\s*(\d+[\.\)])\s*(.*)$"
        )

        # Citation graph extraction regex
        self.legal_basis_line_pattern = re.compile(r"(?im)^\s*căn cứ\b.*$")
        self.legal_ref_pattern = re.compile(
            r"(?i)\b(Hiến pháp|Bộ luật|Luật|Nghị quyết|Pháp lệnh|Nghị định|Thông tư)\b([^.;\n]*)"
        )

    @staticmethod
    def _slugify(value: str) -> str:
        return re.sub(r"[^a-z0-9]+", "-", (value or "").lower()).strip("-") or "unknown"

    @staticmethod
    def _canonical_doc_type(raw: str) -> str:
        text = (raw or "").lower()
        if "hiến pháp" in text or "hien phap" in text:
            return "constitution"
        if "bộ luật" in text or "bo luat" in text:
            return "code"
        if "luật" in text or "luat" in text:
            return "law"
        if "pháp lệnh" in text or "phap lenh" in text:
            return "ordinance"
        if "nghị quyết" in text or "nghi quyet" in text:
            return "resolution"
        if "nghị định" in text or "nghi dinh" in text:
            return "decree"
        if "thông tư" in text or "thong tu" in text:
            return "circular"
        return "other"

    @staticmethod
    def _extract_year(text: str) -> str:
        match = re.search(r"\b(19|20)\d{2}\b", text or "")
        return match.group(0) if match else ""

    @staticmethod
    def _extract_doc_number(text: str) -> str:
        number_patterns = [
            r"(?i)(?:số\s*)?(\d+\/\d+(?:\/[A-ZĐ\-]+)?)",
            r"(?i)(\d+\/[A-ZĐ\-]+)",
        ]
        for pattern in number_patterns:
            match = re.search(pattern, text or "")
            if match:
                return compact_whitespace(match.group(1))
        return ""

    @staticmethod
    def _extract_first_sentence(text: str) -> str:
        """Tóm tắt nội dung điều/khoản bằng LLM (Groq) hoặc Regex làm fallback."""
        if not text:
            return ""

        # 1. Gọi Groq API để tóm tắt thông minh, nhẹ - nhanh
        try:
            import requests
            import os
            api_key = os.environ.get("LLM_API_KEY", "<YOUR_GROQ_API_KEY>")
            base_url = os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1")

            # Sử dụng model Groq siêu nhẹ
            model = os.environ.get("NOTEBOOK_LLM_MODEL", "llama-3.1-8b-instant")

            headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
            payload = {
                "model": model,
                "messages": [
                    {"role": "system", "content": "Bạn là AI tóm tắt luật. Viết tóm tắt đoạn luật sau thành một câu ngắn gọn (dưới 100 từ). KHÔNG giải thích."},
                    {"role": "user", "content": text[:3000]}
                ],
                "max_tokens": 100,
                "temperature": 0.0
            }
            # CÀI ĐẶT TIMEOUT = 10s THEO YÊU CẦU
            res = requests.post(f"{base_url.rstrip('/')}/chat/completions", json=payload, headers=headers, timeout=10.0)
            if res.status_code == 200:
                summary = res.json()["choices"][0]["message"]["content"].strip()
                return summary[:500]
        except Exception:
            pass

        # 2. Fallback Regex (hoặc khi mất quá 10s): Match câu đầu tiên kết thúc bằng . ! ?
        match = re.match(r"^([^.!?\n]+[.!?]?)", text.strip())
        if match:
            summary = match.group(1).strip()
            return summary[:250] + "..." if len(summary) > 250 else summary

        sliced = text.strip()[:250]
        return sliced + "..." if len(text.strip()) > 250 else sliced

    def _build_document_uid(self, metadata: dict) -> str:
        legal_type = self._canonical_doc_type(metadata.get("legal_type", ""))
        doc_number = metadata.get("document_number") or metadata.get("id") or "unknown"
        issue_date = str(metadata.get("issue_date") or metadata.get("issued_date") or "")
        year = self._extract_year(issue_date) or self._extract_year(str(metadata.get("title", ""))) or "unknown"
        return f"doc::{legal_type}::{self._slugify(str(doc_number))}::{year}"

    def _build_parent_law_id(self, doc_type: str, doc_number: str, year: str, doc_title: str) -> str:
        basis = doc_number or doc_title or "unknown"
        return f"parent::{doc_type}::{self._slugify(basis)}::{year or 'unknown'}"

    def _parse_legal_basis_line(self, raw_line: str):
        references = []
        line = compact_whitespace(raw_line)
        for match in self.legal_ref_pattern.finditer(line):
            raw_type = compact_whitespace(match.group(1))
            tail = compact_whitespace(match.group(2))
            full_ref = compact_whitespace(f"{raw_type} {tail}")
            doc_type = self._canonical_doc_type(raw_type)
            year = self._extract_year(full_ref)
            doc_number = self._extract_doc_number(full_ref)
            parent_law_id = self._build_parent_law_id(
                doc_type=doc_type,
                doc_number=doc_number,
                year=year,
                doc_title=full_ref,
            )
            references.append(
                {
                    "basis_line": line,
                    "doc_type": doc_type,
                    "doc_number": doc_number,
                    "doc_year": year,
                    "doc_title": full_ref,
                    "parent_law_id": parent_law_id,
                }
            )
        return references

    def _extract_legal_basis_metadata(self, content: str, metadata: dict):
        preamble = "\n".join((content or "").splitlines()[:80])
        legal_basis_refs = []

        for line in preamble.splitlines():
            if self.legal_basis_line_pattern.match(line):
                legal_basis_refs.extend(self._parse_legal_basis_line(line))

        dedup_refs = []
        seen = set()
        for ref in legal_basis_refs:
            key = (ref.get("parent_law_id"), ref.get("doc_title"))
            if key in seen:
                continue
            seen.add(key)
            dedup_refs.append(ref)

        legal_type = self._canonical_doc_type(metadata.get("legal_type", ""))
        parent_law_candidates = [r for r in dedup_refs if r.get("doc_type") in {"law", "code", "constitution", "ordinance"}]
        parent_law_ids = [r.get("parent_law_id") for r in parent_law_candidates if r.get("parent_law_id")]

        parent_law_id = None
        if legal_type in {"decree", "circular"} and parent_law_ids:
            parent_law_id = parent_law_ids[0]

        document_uid = self._build_document_uid(metadata)
        citation_graph_edges = [
            {
                "edge_type": "LEGAL_BASIS",
                "from_document_uid": document_uid,
                "to_parent_law_id": pid,
            }
            for pid in parent_law_ids
        ]

        return {
            "document_uid": document_uid,
            "legal_basis_texts": sorted({r.get("basis_line", "") for r in dedup_refs if r.get("basis_line")}),
            "legal_basis_refs": dedup_refs,
            "parent_law_id": parent_law_id,
            "parent_law_ids": sorted(set(parent_law_ids)),
            "citation_graph_edges": citation_graph_edges,
        }

    def _normalize_metadata(self, metadata):
        data = dict(metadata)
        data["id"] = str(data.get("id", ""))
        data["legal_sectors_list"] = split_sector_list(data.get("legal_sectors"))
        data["legal_sectors_text"] = "; ".join(data["legal_sectors_list"])
        data["signer"] = data.get("signer") or data.get("signers") or ""
        data["document_uid"] = self._build_document_uid(data)

        # ======================
        # SỬA Ở ĐÂY: Thêm trường url
        # ======================
        data["url"] = str(data.get("url") or data.get("link") or "")

        return data

    def _make_breadcrumb(
        self,
        chapter_ref=None,
        article_ref=None,
        clause_ref=None,
        point_ref=None,
        is_appendix=False,
        section_label=None,
    ):
        parts = []
        if section_label:
            parts.append(section_label)
        if chapter_ref:
            parts.append(chapter_ref)
        if article_ref:
            parts.append(article_ref)
        if clause_ref:
            parts.append(clause_ref)
        if point_ref:
            parts.append(point_ref)
        if is_appendix and not section_label:
            parts.append("Phu luc")

        breadcrumb_path = " > ".join(parts) if parts else "Noi dung chung"
        return {
            "breadcrumb_path": breadcrumb_path,
            "breadcrumb_level_1": parts[0] if len(parts) > 0 else None,
            "breadcrumb_level_2": parts[1] if len(parts) > 1 else None,
            "breadcrumb_level_3": parts[2] if len(parts) > 2 else None,
            "breadcrumb_level_4": parts[3] if len(parts) > 3 else None,
        }

    def _build_metadata_header(self, metadata, is_appendix, article_ref=None, clause_ref=None, chapter_ref=None, point_ref=None, breadcrumb_path=None):
        header_lines = [
            "[LEGAL HEADER]",
            f"- Title: {metadata.get('title', 'N/A')}",
            f"- Document number: {metadata.get('document_number', 'N/A')}",
            f"- Legal type: {metadata.get('legal_type', 'N/A')}",
            f"- Issuing authority: {metadata.get('issuing_authority', 'N/A')}",
            f"- Legal sectors: {metadata.get('legal_sectors_text', metadata.get('legal_sectors', 'N/A'))}",
            f"- Breadcrumb: {breadcrumb_path or 'N/A'}",
            f"- Chapter: {chapter_ref or 'N/A'}",
            f"- Article: {article_ref or 'N/A'}",
            f"- Clause: {clause_ref or 'N/A'}",
            f"- Point: {point_ref or 'N/A'}",
            f"- Is appendix: {is_appendix}",
            f"- Source URL: {metadata.get('url', 'N/A')}",
            "",
        ]
        return "\n".join(header_lines)

    def _build_citation(self, metadata, chapter_ref, article_ref, clause_ref, is_appendix, appendix_part=None, point_ref=None):
        pieces = [metadata.get("document_number") or metadata.get("title") or "Van ban"]
        if chapter_ref:
            pieces.append(chapter_ref)
        if article_ref:
            pieces.append(article_ref)
        if clause_ref:
            pieces.append(clause_ref)
        if point_ref:
            pieces.append(point_ref)
        if is_appendix and appendix_part:
            pieces.append(appendix_part)
        return " | ".join([piece for piece in pieces if piece])

    def _semantic_split_intro(self, text: str) -> List[str]:
        """Tách phần Căn cứ thay vì chia đều theo độ dài kí tự."""
        lines = text.splitlines()
        chunks = []
        cur_chunk = []
        for line in lines:
            if re.match(r"(?i)^\s*(căn cứ|chiếu theo|theo quy định|luật)\b", line):
                if cur_chunk:
                    chunks.append("\n".join(cur_chunk).strip())
                    cur_chunk = []
            cur_chunk.append(line)
        if cur_chunk:
            chunks.append("\n".join(cur_chunk).strip())
        return [c for c in chunks if c.strip()]

    def _semantic_split_appendix(self, text: str) -> List[str]:
        """Tách phụ lục theo các mục I., II., 1., 2., a), b) thay vì đếm kí tự."""
        pattern = re.compile(r"(?im)^\s*(?:[IVXLCDM]+\.|[0-9]+\.|[a-z]\)|[-+])\s+")
        lines = text.splitlines()
        chunks = []
        cur_chunk = []
        for line in lines:
            if pattern.match(line):
                if cur_chunk:
                    chunks.append("\n".join(cur_chunk).strip())
                    cur_chunk = []
            cur_chunk.append(line)
        if cur_chunk:
            chunks.append("\n".join(cur_chunk).strip())
        return [c for c in chunks if c.strip()]

    def _split_main_and_appendix(self, content):
        match = self.appendix_pattern.search(content)
        if match:
            return content[: match.start()].strip(), content[match.start() :].strip()
        return content.strip(), ""

    def _split_articles(self, main_text):
        intro_lines = []
        articles = []
        current_article = None
        current_chapter = None

        for raw_line in main_text.splitlines():
            line = raw_line.rstrip()

            if not line.strip():
                if current_article is None:
                    intro_lines.append("")
                else:
                    current_article["lines"].append("")
                continue

            chapter_match = self.chapter_pattern.match(line)
            if chapter_match:
                current_chapter = compact_whitespace(line)
                if current_article is None:
                    intro_lines.append(line)
                else:
                    current_article["lines"].append(line)
                continue

            article_match = self.article_pattern.match(line)
            if article_match:
                if current_article is not None:
                    articles.append(current_article)

                current_article = {
                    "chapter_ref": current_chapter,
                    "article_ref": compact_whitespace(article_match.group(1)),
                    "article_title": compact_whitespace(article_match.group(2)),
                    "lines": [line],
                }
                continue

            if current_article is None:
                intro_lines.append(line)
            else:
                current_article["lines"].append(line)

        if current_article is not None:
            articles.append(current_article)

        return "\n".join(intro_lines).strip(), articles

    def _split_clauses(self, article):
        full_article_text = "\n".join(article["lines"]).strip()
        body_lines = article["lines"][1:] if len(article["lines"]) > 1 else []
        clauses = []
        current_clause = None

        for raw_line in body_lines:
            line = raw_line.rstrip()
            clause_match = self.clause_pattern.match(line)

            if clause_match:
                if current_clause is not None:
                    clauses.append(current_clause)
                clause_ref = compact_whitespace(clause_match.group(1) or clause_match.group(3))
                clause_tail = compact_whitespace(clause_match.group(2) or clause_match.group(4))
                current_clause = {"clause_ref": clause_ref, "lines": [f"{clause_ref} {clause_tail}".strip()]}
                continue

            if current_clause is None:
                if line.strip():
                    current_clause = {"clause_ref": None, "lines": [line]}
            else:
                current_clause["lines"].append(line)

        if current_clause is not None:
            clauses.append(current_clause)

        if not clauses:
            clauses = [{"clause_ref": None, "lines": article["lines"]}]

        return full_article_text, clauses

    def _appendix_chunks(self, appendix_text, metadata, doc_id, graph_meta):
        chunks = []
        if not appendix_text:
            return chunks

        app_chunks = self._semantic_split_appendix(appendix_text)
        for part_idx, chunk_text in enumerate(app_chunks, start=1):
            breadcrumb = self._make_breadcrumb(
                is_appendix=True,
                section_label="Phu luc",
                article_ref="Phu luc",
                clause_ref=f"P{part_idx}",
            )
            header_app = self._build_metadata_header(
                metadata=metadata,
                is_appendix=True,
                article_ref="Phu luc",
                clause_ref=f"P{part_idx}",
                breadcrumb_path=breadcrumb["breadcrumb_path"],
            )
            citation = self._build_citation(
                metadata=metadata,
                chapter_ref=None,
                article_ref="Phu luc",
                clause_ref=None,
                is_appendix=True,
                appendix_part=f"P{part_idx}",
            )
            chunks.append(
                {
                    "chunk_id": f"{doc_id}::appendix::{part_idx}",
                    "text_to_embed": (
                        f"{header_app}[BREADCRUMB] {breadcrumb['breadcrumb_path']}\n"
                        f"[PHAN {part_idx}]\n{chunk_text.strip()}"
                    ),
                    "unit_text": chunk_text.strip(),
                    "metadata": {
                        **metadata,
                        **graph_meta,
                        **breadcrumb,
                        "is_appendix": True,
                        "chapter_ref": None,
                        "article_id": f"{doc_id}::appendix::{part_idx}",
                        "article_ref": "Phu luc",
                        "article_title": "Phu luc",
                        "clause_ref": f"P{part_idx}",
                        "point_ref": None,
                        "parent_article_text": chunk_text.strip(),
                        "reference_citation": citation,
                    },
                }
            )
        return chunks

    def process_document(self, content, metadata):
        metadata = self._normalize_metadata(metadata)
        content = str(content or "").replace("\r\n", "\n").strip()
        doc_id = metadata.get("id") or str(uuid.uuid4())
        graph_meta = self._extract_legal_basis_metadata(content=content, metadata=metadata)
        chunks = []

        main_text, appendix_text = self._split_main_and_appendix(content)
        intro_text, articles = self._split_articles(main_text)

        if intro_text:
            intro_sub_chunks = self._semantic_split_intro(intro_text)
            for idx, sub_text in enumerate(intro_sub_chunks):
                article_id = f"{doc_id}::preamble"
                ref_tag = "Noi dung" if len(intro_sub_chunks) == 1 else f"Noi dung (P{idx+1})"
                breadcrumb = self._make_breadcrumb(section_label="Mo dau", article_ref=ref_tag)

                citation = self._build_citation(
                    metadata=metadata,
                    chapter_ref=None,
                    article_ref=ref_tag,
                    clause_ref=None,
                    is_appendix=False,
                )
                header = self._build_metadata_header(
                    metadata=metadata,
                    is_appendix=False,
                    article_ref=ref_tag,
                    breadcrumb_path=breadcrumb["breadcrumb_path"],
                )
                chunks.append(
                    {
                        "chunk_id": f"{article_id}::{idx}",
                        "text_to_embed": (
                            f"{header}[BREADCRUMB] {breadcrumb['breadcrumb_path']}\n"
                            f"[NOI DUNG]\n{sub_text}"
                        ),
                        "unit_text": sub_text,
                        "metadata": {
                            **metadata,
                            **graph_meta,
                            **breadcrumb,
                            "is_appendix": False,
                            "chapter_ref": None,
                            "article_id": article_id,
                            "article_ref": ref_tag,
                            "article_title": "Noi dung",
                            "clause_ref": None,
                            "point_ref": None,
                            "parent_article_text": sub_text,
                            "reference_citation": citation,
                        },
                    }
                )

        for article_index, article in enumerate(articles, start=1):
            article_id = f"{doc_id}::article::{article_index}"
            full_article_text, clause_entries = self._split_clauses(article)
            article_context = (
                f"{article['chapter_ref']}\n{full_article_text}".strip()
                if article.get("chapter_ref")
                else full_article_text
            )

            for clause_index, clause in enumerate(clause_entries, start=1):
                clause_ref = clause["clause_ref"]
                unit_text = "\n".join(clause["lines"]).strip()
                if not unit_text:
                    continue

                breadcrumb = self._make_breadcrumb(
                    chapter_ref=article.get("chapter_ref"),
                    article_ref=article["article_ref"],
                    clause_ref=clause_ref,
                )

                citation = self._build_citation(
                    metadata=metadata,
                    chapter_ref=article.get("chapter_ref"),
                    article_ref=article["article_ref"],
                    clause_ref=clause_ref,
                    is_appendix=False,
                    point_ref=None,
                )
                header = self._build_metadata_header(
                    metadata=metadata,
                    is_appendix=False,
                    article_ref=article["article_ref"],
                    clause_ref=clause_ref,
                    chapter_ref=article.get("chapter_ref"),
                    point_ref=None,
                    breadcrumb_path=breadcrumb["breadcrumb_path"],
                )

                chunks.append(
                    {
                        "chunk_id": f"{article_id}::c{clause_index}",
                        "text_to_embed": (
                            f"{header}[BREADCRUMB] {breadcrumb['breadcrumb_path']}\n"
                            f"[NOI DUNG DIEU/KHOAN]\n{unit_text}"
                        ),
                        "unit_text": unit_text,
                        "metadata": {
                            **metadata,
                            **graph_meta,
                            **breadcrumb,
                            "is_appendix": False,
                            "chapter_ref": article.get("chapter_ref"),
                            "article_id": article_id,
                            "article_ref": article["article_ref"],
                            "article_title": article.get("article_title"),
                            "clause_ref": clause_ref,
                            "point_ref": None,
                            "parent_article_text": article_context,
                            "reference_citation": citation,
                        },
                    }
                )

        chunks.extend(self._appendix_chunks(appendix_text, metadata, doc_id, graph_meta))
        return chunks


chunker = AdvancedLegalChunker()
result_chunks = chunker.process_document(content_raw, metadata_raw)

print(f"KIEM TRA document id: {target_id}")
print(f"Tong so chunk sinh ra: {len(result_chunks)}")
print("-" * 140)

if result_chunks:
    first_chunk = result_chunks[0]
    meta = first_chunk["metadata"]

    print("=== CHI TIẾT CHUNK ĐẦU TIÊN ===")
    print(f"1. Chunk ID:          {first_chunk['chunk_id']}")
    print(f"2. Breadcrumb Path:   {meta.get('breadcrumb_path')}")
    print(f"3. Citation:          {meta.get('reference_citation')}")
    print(f"4. Is Appendix:       {meta.get('is_appendix')}")
    print(f"5. Parent Law IDs:    {meta.get('parent_law_ids')}")
    print(f"6. Source URL:        {meta.get('url', '')}")
    print(f"7. Chiều dài (chars): {len(first_chunk['text_to_embed'])} ký tự\n")

    print("[TEXT_TO_EMBED FULL (Phần chữ đưa vào model Vector BGE-M3)]")
    print("-" * 80)
    print(first_chunk["text_to_embed"])
    print("-" * 80)

    print("\n[UNIT_TEXT (Phần nội dung lõi của Điều/Khoản)]")
    print("-" * 80)
    print(first_chunk["unit_text"])
    print("-" * 80)

    print("\n[PARENT_ARTICLE_TEXT (Ngữ cảnh Điều cha)]")
    print("-" * 80)
    print(meta.get("parent_article_text", ""))
    print("-" * 80)
else:
    print("Không có chunk nào được tạo ra từ văn bản này!")

KIEM TRA document id: 693036
Tong so chunk sinh ra: 10
--------------------------------------------------------------------------------------------------------------------------------------------
=== CHI TIẾT CHUNK ĐẦU TIÊN ===
1. Chunk ID:          693036::preamble::0
2. Breadcrumb Path:   Mo dau > Noi dung (P1)
3. Citation:          115/NQ-HĐBCQG | Noi dung (P1)
4. Is Appendix:       False
5. Parent Law IDs:    ['parent::constitution::203-2025-qh::2025', 'parent::law::85-2015-qh::2015']
6. Source URL:        https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Nghi-quyet-115-NQ-HDBCQG-2026-khuyet-nguoi-ung-cu-dai-bieu-Hoi-dong-nhan-dan-ly-do-bat-kha-khang-693036.aspx
7. Chiều dài (chars): 959 ký tự

[TEXT_TO_EMBED FULL (Phần chữ đưa vào model Vector BGE-M3)]
--------------------------------------------------------------------------------
[LEGAL HEADER]
- Title: Nghị quyết 115/NQ-HĐBCQG năm 2026 hướng dẫn xử lý trường hợp khuyết người ứng cử đại biểu Hội đồng nhân dân vì lý do bất khả

In [8]:
# # Cell 5.1: In chi tiet cau truc (Anatomy) cua cac chunk dau tien
# import json

# print(f"=== CHI TIET KET QUA CHUNKING CHO VAN BAN: {target_id} ===")
# print(f"Tong so chunk duoc tao ra: {len(result_chunks)}\n")

# # Lay 2 chunk dau tien de kiem tra
# chunks_to_inspect = result_chunks[:2]

# for i, chunk in enumerate(chunks_to_inspect):
#     print("=" * 100)
#     print(f" CHUNK [{i}] - {chunk['reference_tag']}")
#     print("=" * 100)

#     # 1. Hien thi Text to Embed
#     print("\n[1] VAN BAN DE NHUNG (Text to Embed):")
#     print("-" * 60)
#     print(chunk["text_to_embed"])
#     print("-" * 60)

#     # 2. Hien thi Unit Text
#     print("\n[2] VAN BAN MANH (Unit Text / Child Chunk):")
#     print("-" * 60)
#     print(chunk["unit_text"])
#     print("-" * 60)

#     # 3. Hien thi Parent Context
#     print("\n[3] NGU CANH DIEU LUAT (Parent Article Text):")
#     print("-" * 60)
#     print(chunk["metadata"].get("parent_article_text", "Khong co"))
#     print("-" * 60)

#     # 4. Hien thi Metadata cho Qdrant Payload
#     print("\n[4] METADATA (Qdrant Payload):")
#     print("-" * 60)
#     meta = chunk["metadata"]
#     meta_preview = {
#         "document_uid": meta.get("document_uid"),
#         "article_id": meta.get("article_id"),
#         "chapter_ref": meta.get("chapter_ref"),
#         "article_ref": meta.get("article_ref"),
#         "clause_ref": meta.get("clause_ref"),
#         "point_ref": meta.get("point_ref"),
#         "breadcrumb_path": meta.get("breadcrumb_path"),
#         "breadcrumb_level_1": meta.get("breadcrumb_level_1"),
#         "breadcrumb_level_2": meta.get("breadcrumb_level_2"),
#         "breadcrumb_level_3": meta.get("breadcrumb_level_3"),
#         "reference_citation": meta.get("reference_citation"),
#         "is_appendix": meta.get("is_appendix"),
#         "legal_sectors_list": meta.get("legal_sectors_list"),
#         "legal_basis_texts": meta.get("legal_basis_texts", [])[:3],
#         "parent_law_id": meta.get("parent_law_id"),
#         "parent_law_ids": meta.get("parent_law_ids", []),
#     }
#     print(json.dumps(meta_preview, indent=4, ensure_ascii=False))
#     print("\n" + "*" * 100 + "\n")

In [9]:
# # Cell 6: config Qdrant tu .env, chunk docs va tao dense+sparse embeddings (CPU)
# import time


# def ensure_hybrid_collection(client: QdrantClient, collection_name: str):
#     if client.collection_exists(collection_name):
#         print(f"Collection already exists: {collection_name}")
#         return

#     client.create_collection(
#         collection_name=collection_name,
#         vectors_config={
#             "dense": models.VectorParams(
#                 size=dense_dim,
#                 distance=models.Distance.COSINE,
#                 on_disk=True,
#             )
#         },
#         sparse_vectors_config={
#             "sparse": models.SparseVectorParams(
#                 index=models.SparseIndexParams(on_disk=False)
#             )
#         },
#     )
#     print(f"Created hybrid collection: {collection_name}")


# ensure_hybrid_collection(qdrant_client, COLLECTION_NAME)

# PIPELINE_STATS = {}
# all_processed_chunks = []
# print(f"Start chunking {len(ds_content_10000)} documents...")

# t_chunk_start = time.perf_counter()
# for row in ds_content_10000:
#     doc_id = str(row["id"])
#     content = row["content"]
#     metadata = metadata_by_id.get(doc_id)
#     if not metadata:
#         continue
#     all_processed_chunks.extend(chunker.process_document(content, metadata))
# PIPELINE_STATS["chunk_seconds"] = time.perf_counter() - t_chunk_start

# print(f"Total chunks: {len(all_processed_chunks)}")

# all_texts = [chunk["text_to_embed"] for chunk in all_processed_chunks]

# print("Encoding dense vectors with BGE-M3...")
# import torch
# batch_size = 32 if torch.cuda.is_available() else 4  # conservative default for CPU

# t_dense_start = time.perf_counter()
# all_dense_vectors = embedder.encode_dense(all_texts, batch_size=batch_size)
# PIPELINE_STATS["dense_seconds"] = time.perf_counter() - t_dense_start

# print("Encoding sparse vectors with BGE-M3 lexical weights...")
# t_sparse_start = time.perf_counter()
# all_sparse_vectors = sparse_encoder.encode_sparse_documents(all_texts, batch_size=batch_size)
# PIPELINE_STATS["sparse_seconds"] = time.perf_counter() - t_sparse_start

# PIPELINE_STATS["documents"] = len(ds_content_10000)
# PIPELINE_STATS["chunks"] = len(all_processed_chunks)
# print("Dense + sparse vectors ready for hybrid indexing.")
# print(PIPELINE_STATS)

In [10]:
# # Cell 7: build record (dense+sparse+payload), upsert to Qdrant, payload indexes, va benchmark indexing
# import time
# import uuid

# # =========================================================================
# # Kiem tra va tao lai collection neu chua ton tai
# # =========================================================================
# if not qdrant_client.collection_exists(COLLECTION_NAME):
#     print(f"Collection '{COLLECTION_NAME}' chua ton tai. Dang tao moi...")
#     qdrant_client.create_collection(
#         collection_name=COLLECTION_NAME,
#         vectors_config={
#             "dense": models.VectorParams(
#                 size=dense_dim,
#                 distance=models.Distance.COSINE,
#                 on_disk=True,
#             )
#         },
#         sparse_vectors_config={
#             "sparse": models.SparseVectorParams(
#                 index=models.SparseIndexParams(on_disk=False)
#             )
#         },
#     )
#     print(f"Da tao collection: {COLLECTION_NAME}")
# else:
#     print(f"Collection '{COLLECTION_NAME}' da ton tai.")
# # =========================================================================

# # Build unified records: moi chunk luu dong thoi dense vector + sparse vector + payload
# index_records = []
# t_point_build_start = time.perf_counter()

# for idx, chunk in enumerate(all_processed_chunks):
#     meta = chunk["metadata"]
#     payload = {
#         "document_id": str(meta.get("id", "")),
#         "document_uid": meta.get("document_uid"),
#         "chunk_id": chunk["chunk_id"],
#         "document_number": meta.get("document_number", "N/A"),
#         "title": meta.get("title", "N/A"),
#         "legal_type": meta.get("legal_type", "N/A"),
#         "legal_sectors": meta.get("legal_sectors_list", []),
#         "issuing_authority": meta.get("issuing_authority", "N/A"),
#         "signer": meta.get("signer", meta.get("signers", "N/A")),
#         "chapter_ref": meta.get("chapter_ref"),
#         "article_id": meta.get("article_id"),
#         "article_ref": meta.get("article_ref"),
#         "article_title": meta.get("article_title"),
#         "clause_ref": meta.get("clause_ref"),
#         "point_ref": meta.get("point_ref"),
#         "reference_tag": chunk["reference_tag"],
#         "reference_citation": meta.get("reference_citation"),
#         "is_appendix": bool(meta.get("is_appendix", False)),
#         "parent_article_text": meta.get("parent_article_text", ""),
#         "matched_clause_text": chunk.get("unit_text", ""),
#         "chunk_text": chunk["text_to_embed"],

#         # Regex tree breadcrumb metadata
#         "breadcrumb_path": meta.get("breadcrumb_path"),
#         "breadcrumb_level_1": meta.get("breadcrumb_level_1"),
#         "breadcrumb_level_2": meta.get("breadcrumb_level_2"),
#         "breadcrumb_level_3": meta.get("breadcrumb_level_3"),
#         "breadcrumb_level_4": meta.get("breadcrumb_level_4"),

#         # Citation Graph ingestion metadata (Parent Law bridge)
#         "legal_basis_texts": meta.get("legal_basis_texts", []),
#         "legal_basis_refs": meta.get("legal_basis_refs", []),
#         "parent_law_id": meta.get("parent_law_id"),
#         "parent_law_ids": meta.get("parent_law_ids", []),
#         "citation_graph_edges": meta.get("citation_graph_edges", []),
#     }

#     index_records.append(
#         {
#             "qdrant_id": str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"])),
#             "chunk_id": chunk["chunk_id"],
#             "dense_vector": all_dense_vectors[idx],
#             "sparse_vector": all_sparse_vectors[idx],
#             "payload": payload,
#         }
#     )

# PIPELINE_STATS["point_build_seconds"] = time.perf_counter() - t_point_build_start

# points_to_insert = [
#     models.PointStruct(
#         id=record["qdrant_id"],
#         vector={
#             "dense": record["dense_vector"],
#             "sparse": record["sparse_vector"],
#         },
#         payload=record["payload"],
#     )
#     for record in index_records
# ]

# BATCH_UPSERT = 100
# print(f"Upserting {len(points_to_insert)} hybrid points to Qdrant ({os.environ.get('QDRANT_ACTIVE_MODE', 'unknown')})...")

# t_upsert_start = time.perf_counter()
# for start in range(0, len(points_to_insert), BATCH_UPSERT):
#     batch = points_to_insert[start : start + BATCH_UPSERT]
#     qdrant_client.upsert(
#         collection_name=COLLECTION_NAME,
#         points=batch,
#         wait=True,
#     )
# PIPELINE_STATS["upsert_seconds"] = time.perf_counter() - t_upsert_start

# print(f"Done upsert {len(points_to_insert)} points to collection {COLLECTION_NAME}.")


# def safe_create_payload_index(client, collection_name, field_name, field_schema):
#     try:
#         client.create_payload_index(
#             collection_name=collection_name,
#             field_name=field_name,
#             field_schema=field_schema,
#             wait=True,
#         )
#         print(f"[index] Created: {field_name}")
#     except Exception as exc:
#         print(f"[index] Skipped/Exists: {field_name} - {exc}")


# print("\n--- Setting up Payload Indexes ---")
# for field in [
#     "document_id",
#     "document_uid",
#     "document_number",
#     "chapter_ref",
#     "article_ref",
#     "clause_ref",
#     "point_ref",
#     "parent_law_id",
#     "parent_law_ids",
#     "breadcrumb_level_1",
#     "breadcrumb_level_2",
#     "breadcrumb_level_3",
#     "breadcrumb_level_4",
# ]:
#     safe_create_payload_index(qdrant_client, COLLECTION_NAME, field, models.PayloadSchemaType.KEYWORD)

# for field in ["legal_type", "legal_sectors", "issuing_authority", "signer"]:
#     safe_create_payload_index(qdrant_client, COLLECTION_NAME, field, models.PayloadSchemaType.KEYWORD)

# safe_create_payload_index(qdrant_client, COLLECTION_NAME, "is_appendix", models.PayloadSchemaType.BOOL)

# safe_create_payload_index(
#     qdrant_client,
#     COLLECTION_NAME,
#     "title",
#     models.TextIndexParams(
#         type="text",
#         tokenizer=models.TokenizerType.WORD,
#         min_token_len=2,
#         max_token_len=20,
#         lowercase=True,
#     ),
# )

# safe_create_payload_index(
#     qdrant_client,
#     COLLECTION_NAME,
#     "breadcrumb_path",
#     models.TextIndexParams(
#         type="text",
#         tokenizer=models.TokenizerType.WORD,
#         min_token_len=2,
#         max_token_len=30,
#         lowercase=True,
#     ),
# )

# try:
#     qdrant_client.update_collection(
#         collection_name=COLLECTION_NAME,
#         quantization_config=models.ScalarQuantization(
#             scalar=models.ScalarQuantizationConfig(
#                 type="int8",
#                 quantile=0.99,
#                 always_ram=True,
#             )
#         ),
#     )
#     print("\n[Quantization] Enabled ScalarQuantization int8 for dense vectors.")
# except Exception as exc:
#     print(f"\n[Quantization] Skip update: {exc}")

# # =========================
# # Benchmark indexing metrics
# # =========================
# indexing_total_seconds = sum(
#     PIPELINE_STATS.get(k, 0.0)
#     for k in ["chunk_seconds", "dense_seconds", "sparse_seconds", "point_build_seconds", "upsert_seconds"]
# )
# PIPELINE_STATS["total_seconds_measured"] = indexing_total_seconds

# total_chunks = int(PIPELINE_STATS.get("chunks", 0))
# throughput = (total_chunks / indexing_total_seconds) if indexing_total_seconds > 0 else 0.0
# PIPELINE_STATS["indexing_throughput_chunks_per_sec"] = throughput

# print("\n=== INDEXING BENCHMARK ===")
# print(f"Total chunks: {total_chunks}")
# print(f"Total indexing time (s): {indexing_total_seconds:,.4f}")
# print(f"Throughput (chunks/s): {throughput:,.4f}")
# print("\nPIPELINE_STATS:", PIPELINE_STATS)

In [11]:
# Cell 6+7: Chunking, Embedding, Upserting & Benchmarking (Memory Optimized + DEBUG LOGS)
import time
import uuid
import gc
import torch
from qdrant_client import models

# =========================================================================
# 1. KIỂM TRA VÀ TẠO COLLECTION (NẾU CHƯA TỒN TẠI)
# =========================================================================
if not qdrant_client.collection_exists(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' chua ton tai. Dang tao moi...")
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
                on_disk=True,
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        },
    )
    print(f"Da tao collection: {COLLECTION_NAME}")
else:
    print(f"Collection '{COLLECTION_NAME}' da ton tai.")

# =========================================================================
# 2. KHỞI TẠO BIẾN THỐNG KÊ VÀ CẤU HÌNH LÔ (BATCH)
# =========================================================================
PIPELINE_STATS = {
    "chunk_seconds": 0.0,
    "dense_seconds": 0.0,
    "sparse_seconds": 0.0,
    "point_build_seconds": 0.0,
    "upsert_seconds": 0.0,
    "chunks": 0,
    "documents": len(ds_content_10000)
}

BATCH_SIZE_DOCS = 500  # Xử lý 500 văn bản mỗi mẻ để xả RAM liên tục
encode_batch_size = 256 if torch.cuda.is_available() else 16

print(f"Start processing {PIPELINE_STATS['documents']} documents in batches of {BATCH_SIZE_DOCS}...")

# =========================================================================
# 3. VÒNG LẶP XỬ LÝ THEO LÔ (STREAMING)
# =========================================================================
batch_chunks = []
processed_docs_count = 0

for row in ds_content_10000:
    doc_id = str(row["id"])
    content = row["content"]
    metadata = metadata_by_id.get(doc_id)

    if not metadata:
        continue

    # Chunking
    t0 = time.perf_counter()
    batch_chunks.extend(chunker.process_document(content, metadata))
    PIPELINE_STATS["chunk_seconds"] += (time.perf_counter() - t0)

    processed_docs_count += 1

    # Khi gom đủ 1 lô hoặc duyệt đến văn bản cuối cùng thì tiến hành nhúng và upsert
    if processed_docs_count % BATCH_SIZE_DOCS == 0 or processed_docs_count == PIPELINE_STATS["documents"]:
        if not batch_chunks:
            continue

        print(f"\n[+] Đang xử lý lô {processed_docs_count}/{PIPELINE_STATS['documents']} docs ({len(batch_chunks)} chunks)...", flush=True)

        # Lấy text để embed
        batch_texts = [chunk["text_to_embed"] for chunk in batch_chunks]

        # ---------------------------------------------------------
        # DEBUG 1: Theo dõi quá trình Embedding
        # ---------------------------------------------------------
        print(f"    -> Đang chạy Embedding (Dense + Sparse) với batch_size={encode_batch_size}...", end="", flush=True)
        t0 = time.perf_counter()
        batch_dense_vectors, batch_sparse_vectors = hybrid_encoder.encode_hybrid(batch_texts, batch_size=encode_batch_size)
        t_embed = (time.perf_counter() - t0)
        print(f" Xong! ({t_embed:.2f}s)", flush=True)

        PIPELINE_STATS["dense_seconds"] += t_embed / 2
        PIPELINE_STATS["sparse_seconds"] += t_embed / 2

        # ---------------------------------------------------------
        # DEBUG 2: Theo dõi quá trình Build Payload
        # ---------------------------------------------------------
        print("    -> Đang build Payload & Points...", end="", flush=True)
        t0 = time.perf_counter()
        points_to_insert = []
        for idx, chunk in enumerate(batch_chunks):
            meta = chunk["metadata"]

            payload = {
                "document_id": str(meta.get("id", "")),
                "document_uid": meta.get("document_uid"),
                "chunk_id": chunk["chunk_id"],
                "document_number": meta.get("document_number", "N/A"),
                "title": meta.get("title", "N/A"),
                "legal_type": meta.get("legal_type", "N/A"),
                "legal_sectors": meta.get("legal_sectors_list", []),
                "issuing_authority": meta.get("issuing_authority", "N/A"),
                "signer": meta.get("signer") or meta.get("signers", "N/A"),
                # SỬA Ở ĐÂY: Thêm url
                "url": str(meta.get("url", "")),

                # Cấu trúc phân cấp
                "chapter_ref": meta.get("chapter_ref"),
                "article_id": meta.get("article_id"),
                "article_ref": meta.get("article_ref"),
                "clause_ref": meta.get("clause_ref"),
                "point_ref": meta.get("point_ref"),
                "is_appendix": bool(meta.get("is_appendix", False)),
                "breadcrumb_path": meta.get("breadcrumb_path"),
                "reference_citation": meta.get("reference_citation"),

                # Nội dung Text
                "parent_article_text": meta.get("parent_article_text", ""),
                "matched_clause_text": chunk.get("unit_text", ""),
                "chunk_text": chunk["text_to_embed"],
                # "article_summary": chunker._extract_first_sentence(chunk.get("unit_text", "")),

                # Citation Graph
                "legal_basis_texts": meta.get("legal_basis_texts", []),
                "legal_basis_refs": meta.get("legal_basis_refs", []),
                "parent_law_ids": meta.get("parent_law_ids", []),
            }

            qdrant_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"]))
            points_to_insert.append(
                models.PointStruct(
                    id=qdrant_id,
                    vector={
                        "dense": batch_dense_vectors[idx],
                        "sparse": batch_sparse_vectors[idx]
                    },
                    payload=payload,
                )
            )
        t_build = time.perf_counter() - t0
        PIPELINE_STATS["point_build_seconds"] += t_build
        print(f" Xong! ({t_build:.2f}s)", flush=True)

        # ---------------------------------------------------------
        # DEBUG 3: Theo dõi quá trình Upsert lên DB
        # ---------------------------------------------------------
        print("    -> Đang Upsert lên Qdrant...", end="", flush=True)
        t_upsert_start = time.perf_counter()

        CHUNK_UPSERT_SIZE = 40
        total_sub_batches = (len(points_to_insert) + CHUNK_UPSERT_SIZE - 1) // CHUNK_UPSERT_SIZE

        for i, start_idx in enumerate(range(0, len(points_to_insert), CHUNK_UPSERT_SIZE), 1):
            batch_points = points_to_insert[start_idx : start_idx + CHUNK_UPSERT_SIZE]
            try:
                # In tiến độ đè lên dòng hiện tại (carriage return \r)
                print(f"\r    -> Đang Upsert lên Qdrant... ({i}/{total_sub_batches})", end="", flush=True)
                qdrant_client.upsert(
                    collection_name=COLLECTION_NAME,
                    points=batch_points,
                    wait=False
                )
            except Exception as e:
                print(f"\n    [!] Lỗi khi upsert sub-batch {i}: {e}", flush=True)

        t_upsert = time.perf_counter() - t_upsert_start
        PIPELINE_STATS["upsert_seconds"] += t_upsert
        print(f"\r    -> Đang Upsert lên Qdrant... Xong! ({t_upsert:.2f}s)       ", flush=True)

        PIPELINE_STATS["chunks"] += len(batch_chunks)

        # ---------------------------------------------------------
        # DEBUG 4: Theo dõi quá trình giải phóng RAM
        # ---------------------------------------------------------
        print("    -> Đang dọn dẹp RAM (GC)...", end="", flush=True)
        batch_chunks.clear()
        del batch_texts, batch_dense_vectors, batch_sparse_vectors, points_to_insert
        gc.collect()
        print(" Xong!\n", flush=True)

print(f"Đã hoàn tất upsert toàn bộ {PIPELINE_STATS['chunks']} points to collection {COLLECTION_NAME}.")

# =========================================================================
# 4. THIẾT LẬP PAYLOAD INDEXES & QUANTIZATION
# =========================================================================
def safe_create_payload_index(client, collection_name, field_name, field_schema):
    try:
        client.create_payload_index(
            collection_name=collection_name,
            field_name=field_name,
            field_schema=field_schema,
            wait=True,
        )
        print(f"[index] Created: {field_name}")
    except Exception as exc:
        print(f"[index] Skipped/Exists: {field_name} - {exc}")

print("\n--- Setting up Payload Indexes ---")
keyword_fields = [
    "document_id", "document_uid", "document_number", "chapter_ref", "article_ref", "article_id",
    "clause_ref", "point_ref", "parent_law_ids",
    "legal_type", "legal_sectors", "issuing_authority", "signer"
]
for field in keyword_fields:
    safe_create_payload_index(qdrant_client, COLLECTION_NAME, field, models.PayloadSchemaType.KEYWORD)

safe_create_payload_index(qdrant_client, COLLECTION_NAME, "is_appendix", models.PayloadSchemaType.BOOL)

safe_create_payload_index(
    qdrant_client, COLLECTION_NAME, "title",
    models.TextIndexParams(type="text", tokenizer=models.TokenizerType.WORD, min_token_len=2, max_token_len=20, lowercase=True)
)

safe_create_payload_index(
    qdrant_client, COLLECTION_NAME, "breadcrumb_path",
    models.TextIndexParams(type="text", tokenizer=models.TokenizerType.WORD, min_token_len=2, max_token_len=30, lowercase=True)
)

try:
    qdrant_client.update_collection(
        collection_name=COLLECTION_NAME,
        quantization_config=models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(type="int8", quantile=0.99, always_ram=True)
        ),
    )
    print("\n[Quantization] Enabled ScalarQuantization int8 for dense vectors.")
except Exception as exc:
    print(f"\n[Quantization] Skip update: {exc}")

# =========================================================================
# 5. IN CHỈ SỐ BENCHMARK VÀ TÍNH ESTIMATE CHO CÁC MỐC
# =========================================================================
indexing_total_seconds = sum(
    PIPELINE_STATS.get(k, 0.0)
    for k in ["chunk_seconds", "dense_seconds", "sparse_seconds", "point_build_seconds", "upsert_seconds"]
)
PIPELINE_STATS["total_seconds_measured"] = indexing_total_seconds

total_chunks = int(PIPELINE_STATS.get("chunks", 0))
docs_processed = PIPELINE_STATS["documents"]
if docs_processed == 0: docs_processed = 1 # avoid div by zero

throughput = (total_chunks / indexing_total_seconds) if indexing_total_seconds > 0 else 0.0
PIPELINE_STATS["indexing_throughput_chunks_per_sec"] = throughput

avg_per_doc = indexing_total_seconds / docs_processed
avg_per_chunk = indexing_total_seconds / total_chunks if total_chunks > 0 else 0.0

print("\n=== INDEXING BENCHMARK ===")
print(f"Total time:       {indexing_total_seconds:,.2f} s")
print(f"- chunking:       {PIPELINE_STATS['chunk_seconds']:,.2f} s")
print(f"- dense embed:    {PIPELINE_STATS['dense_seconds']:,.2f} s")
print(f"- sparse embed:   {PIPELINE_STATS['sparse_seconds']:,.2f} s")
print(f"- point build:    {PIPELINE_STATS['point_build_seconds']:,.2f} s")
print(f"- upsert:         {PIPELINE_STATS['upsert_seconds']:,.2f} s")
print(f"Avg time / doc:   {avg_per_doc:,.3f} s/doc")
print(f"Avg time / chunk: {avg_per_chunk:,.3f} s/chunk")

print("\n=== ESTIMATES FOR SCALING ===")
docs_levels = [100, 500, 1000, 5000, 10000]
for d in docs_levels:
    est_sec = d * avg_per_doc
    est_min = est_sec / 60
    est_hrs = est_min / 60
    print(f"Estimate for {str(d).rjust(7)} docs: {est_sec:,.1f}s | {est_min:,.1f}m | {est_hrs:,.2f}h")

Collection 'legal_rag_docs_5000' da ton tai.
Start processing 5000 documents in batches of 500...

[+] Đang xử lý lô 500/5000 docs (25782 chunks)...
    -> Đang chạy Embedding (Dense + Sparse) với batch_size=256...

pre tokenize: 100%|██████████| 51/51 [00:09<00:00,  5.35it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Chunks: 100%|██████████| 2/2 [03:53<00:00, 116.83s/it]


 Xong! (237.37s)
    -> Đang build Payload & Points... Xong! (2.55s)
    -> Đang Upsert lên Qdrant... Xong! (208.27s)       
    -> Đang dọn dẹp RAM (GC)... Xong!


[+] Đang xử lý lô 1000/5000 docs (20553 chunks)...
    -> Đang chạy Embedding (Dense + Sparse) với batch_size=256...

Chunks: 100%|██████████| 2/2 [03:14<00:00, 97.22s/it]


 Xong! (197.29s)
    -> Đang build Payload & Points... Xong! (1.08s)
    -> Đang Upsert lên Qdrant... (63/514)
    [!] Lỗi khi upsert sub-batch 63: The read operation timed out
    -> Đang Upsert lên Qdrant... (184/514)
    [!] Lỗi khi upsert sub-batch 184: The read operation timed out
    -> Đang Upsert lên Qdrant... Xong! (154.31s)       
    -> Đang dọn dẹp RAM (GC)... Xong!


[+] Đang xử lý lô 1500/5000 docs (32828 chunks)...
    -> Đang chạy Embedding (Dense + Sparse) với batch_size=256...

...

 Xong! (215.52s)
    -> Đang build Payload & Points... Xong! (2.32s)
    -> Đang Upsert lên Qdrant... Xong! (227.29s)       
    -> Đang dọn dẹp RAM (GC)... Xong!


[+] Đang xử lý lô 4500/5000 docs (26450 chunks)...
    -> Đang chạy Embedding (Dense + Sparse) với batch_size=256...

Chunks: 100%|██████████| 2/2 [04:05<00:00, 122.80s/it]


 Xong! (249.20s)
    -> Đang build Payload & Points... Xong! (2.48s)
    -> Đang Upsert lên Qdrant... Xong! (163.36s)       
    -> Đang dọn dẹp RAM (GC)... Xong!


[+] Đang xử lý lô 5000/5000 docs (24639 chunks)...
    -> Đang chạy Embedding (Dense + Sparse) với batch_size=256...

Chunks: 100%|██████████| 2/2 [03:52<00:00, 116.07s/it]


 Xong! (235.62s)
    -> Đang build Payload & Points... Xong! (2.46s)
    -> Đang Upsert lên Qdrant... (356/616)
    [!] Lỗi khi upsert sub-batch 356: The read operation timed out
    -> Đang Upsert lên Qdrant... Xong! (220.12s)       
    -> Đang dọn dẹp RAM (GC)... Xong!

Đã hoàn tất upsert toàn bộ 242245 points to collection legal_rag_docs_5000.

--- Setting up Payload Indexes ---
[index] Skipped/Exists: document_id - The read operation timed out
[index] Skipped/Exists: document_uid - The read operation timed out
[index] Skipped/Exists: document_number - The read operation timed out
[index] Skipped/Exists: chapter_ref - The read operation timed out
[index] Skipped/Exists: article_ref - The read operation timed out
[index] Skipped/Exists: article_id - The read operation timed out
[index] Skipped/Exists: clause_ref - The read operation timed out
[index] Skipped/Exists: point_ref - The read operation timed out
[index] Skipped/Exists: parent_law_ids - The read operation timed out
[index]

In [12]:
# CPU runtime estimate based on measured pipeline stats
import math

if not PIPELINE_STATS:
    print("PIPELINE_STATS is empty. Please run Cell 6 and Cell 7 first.")
else:
    measured_docs = max(int(PIPELINE_STATS.get("documents", 0)), 1)
    measured_chunks = max(int(PIPELINE_STATS.get("chunks", 0)), 1)

    t_chunk = float(PIPELINE_STATS.get("chunk_seconds", 0.0))
    t_dense = float(PIPELINE_STATS.get("dense_seconds", 0.0))
    t_sparse = float(PIPELINE_STATS.get("sparse_seconds", 0.0))
    t_point_build = float(PIPELINE_STATS.get("point_build_seconds", 0.0))
    t_upsert = float(PIPELINE_STATS.get("upsert_seconds", 0.0))
    t_total = float(PIPELINE_STATS.get("total_seconds_measured", 0.0))

    sec_per_doc = t_total / measured_docs
    sec_per_chunk = t_total / measured_chunks

    for target_docs in [100, 500, 1000, 10000]:
        est_seconds = sec_per_doc * target_docs
        est_minutes = est_seconds / 60.0
        est_hours = est_minutes / 60.0
        print(
            f"Estimate for {target_docs:>5} docs: "
            f"{est_seconds:,.1f}s | {est_minutes:,.1f}m | {est_hours:,.2f}h"
        )

    print("\nMeasured breakdown on CPU:")
    print(f"- chunking:     {t_chunk:,.2f}s")
    print(f"- dense embed:  {t_dense:,.2f}s")
    print(f"- sparse embed: {t_sparse:,.2f}s")
    print(f"- point build:  {t_point_build:,.2f}s")
    print(f"- upsert:       {t_upsert:,.2f}s")
    print(f"- total:        {t_total:,.2f}s")
    print(f"- avg/doc:      {sec_per_doc:,.3f}s")
    print(f"- avg/chunk:    {sec_per_chunk:,.3f}s")

Estimate for   100 docs: 85.8s | 1.4m | 0.02h
Estimate for   500 docs: 429.0s | 7.1m | 0.12h
Estimate for  1000 docs: 857.9s | 14.3m | 0.24h
Estimate for 10000 docs: 8,579.3s | 143.0m | 2.38h

Measured breakdown on CPU:
- chunking:     9.66s
- dense embed:  1,139.80s
- sparse embed: 1,139.80s
- point build:  21.92s
- upsert:       1,978.49s
- total:        4,289.67s
- avg/doc:      0.858s
- avg/chunk:    0.018s


In [13]:
# Cell 8: LegalHybridRetriever 3-step pipeline (Broad Retrieval -> Rerank -> Context Expansion)
from qdrant_client import models
from typing import List, Optional, Any, Dict, Tuple
import re
import time

try:
    from FlagEmbedding import FlagReranker
except Exception:
    FlagReranker = None


def detect_sector_hints(query: str, hot_sectors: List[str]) -> List[str]:
    lowered = query.lower()
    hits = [sector for sector in hot_sectors if sector.lower() in lowered]
    return hits[:3]


def parse_chunk_order(chunk_id: str) -> Tuple[int, int, str]:
    """Sort key for legal chunks in same article.

    Examples:
    - ...::c3 -> (3, 0, ...)
    - ...::c3::p2 -> (3, 2, ...)
    - unknown -> large fallback
    """
    if not chunk_id:
        return (10**9, 10**9, "")

    clause_match = re.search(r"::c(\d+)", chunk_id)
    point_match = re.search(r"::p(\d+)", chunk_id)

    clause_idx = int(clause_match.group(1)) if clause_match else 10**9
    point_idx = int(point_match.group(1)) if point_match else 0
    return (clause_idx, point_idx, chunk_id)


class BGEReranker:
    """Cross-Encoder reranker using BAAI/bge-reranker-v2-m3."""

    def __init__(self, model_name: str = "BAAI/bge-reranker-v2-m3", use_fp16: bool = False):
        self.model_name = model_name
        self.model = None
        self.use_fp16 = use_fp16

    def _lazy_load(self):
        if self.model is None:
            if FlagReranker is None:
                raise RuntimeError(
                    "FlagReranker is unavailable. Please ensure FlagEmbedding is installed correctly."
                )
            self.model = FlagReranker(self.model_name, use_fp16=self.use_fp16)

    def score(self, query: str, docs: List[str]) -> List[float]:
        self._lazy_load()
        if not docs:
            return []
        pairs = [[query, d] for d in docs]
        scores = self.model.compute_score(pairs, normalize=True)
        if isinstance(scores, (int, float)):
            return [float(scores)]
        return [float(s) for s in scores]


class LegalHybridRetriever:
    def __init__(
        self,
        client: QdrantClient,
        collection_name: str,
        hybrid_encoder: Any,
        hot_sectors: List[str],
        reranker: Optional[BGEReranker] = None,
    ):
        self.client = client
        self.collection_name = collection_name
        self.hybrid_encoder = hybrid_encoder
        self.hot_sectors = hot_sectors
        self.reranker = reranker

    def build_filter(
        self,
        query: str,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        must_conditions = []
        should_conditions = []

        if is_appendix is not None:
            must_conditions.append(
                models.FieldCondition(
                    key="is_appendix",
                    match=models.MatchValue(value=is_appendix),
                )
            )
        if legal_type:
            must_conditions.append(
                models.FieldCondition(
                    key="legal_type",
                    match=models.MatchValue(value=legal_type),
                )
            )
        if doc_number:
            must_conditions.append(
                models.FieldCondition(
                    key="document_number",
                    match=models.MatchValue(value=doc_number),
                )
            )

        for sector in detect_sector_hints(query, self.hot_sectors):
            should_conditions.append(
                models.FieldCondition(
                    key="legal_sectors",
                    match=models.MatchValue(value=sector),
                )
            )
            should_conditions.append(
                models.FieldCondition(
                    key="title",
                    match=models.MatchText(text=sector),
                )
            )

        if not must_conditions and not should_conditions:
            return None

        return models.Filter(
            must=must_conditions or None,
            should=should_conditions or None,
        )

    # -----------------------------
    # Step 1: Broad Retrieval
    # -----------------------------
    def broad_retrieve(
        self,
        query: str,
        top_k: int = 12,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        query_filter = self.build_filter(
            query=query,
            is_appendix=is_appendix,
            legal_type=legal_type,
            doc_number=doc_number,
        )

        dense_query = self.hybrid_encoder.encode_dense([query], batch_size=1)[0]
        sparse_query = self.hybrid_encoder.encode_query_sparse(query)

        prefetch_limit = max(top_k * 4, 24)
        raw_hits = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                models.Prefetch(
                    query=dense_query,
                    using="dense",
                    limit=prefetch_limit,
                    filter=query_filter,
                ),
                models.Prefetch(
                    query=sparse_query,
                    using="sparse",
                    limit=prefetch_limit,
                    filter=query_filter,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=prefetch_limit,
            with_payload=True,
        ).points

        # dedup by chunk_id to keep fine-grained candidates for reranking
        dedup = []
        seen_chunk_ids = set()
        for hit in raw_hits:
            payload = hit.payload or {}
            chunk_id = payload.get("chunk_id") or str(hit.id)
            if chunk_id in seen_chunk_ids:
                continue
            seen_chunk_ids.add(chunk_id)
            dedup.append(
                {
                    "id": hit.id,
                    "payload": payload,
                    "rrf_score": float(hit.score),
                    "broad_rank": len(dedup) + 1,
                }
            )
            if len(dedup) >= top_k:
                break

        return dedup

    def _build_rerank_text(self, payload: Dict[str, Any]) -> str:
        title = payload.get("title") or ""
        citation = payload.get("reference_citation") or ""
        clause = payload.get("matched_clause_text") or ""
        parent = payload.get("parent_article_text") or ""
        return f"{title}\n{citation}\n{clause}\n{parent}".strip()

    # -----------------------------
    # Step 2: Cross-Encoder Rerank
    # -----------------------------
    def rerank(self, query: str, candidates: List[Dict[str, Any]], top_l: int = 5):
        if not candidates:
            return []

        if self.reranker is None:
            # fallback: use broad ranking order if reranker unavailable
            return candidates[:top_l]

        docs = [self._build_rerank_text(item["payload"]) for item in candidates]
        rerank_scores = self.reranker.score(query, docs)

        scored = []
        for item, score in zip(candidates, rerank_scores):
            enriched = dict(item)
            enriched["rerank_score"] = float(score)
            scored.append(enriched)

        scored.sort(key=lambda x: x.get("rerank_score", -1e9), reverse=True)
        return scored[:top_l]

    # -----------------------------
    # Step 3: Context Expansion
    # -----------------------------
    def expand_context(self, refined_hits: List[Dict[str, Any]], max_neighbors: int = 8):
        expanded_results = []

        for item in refined_hits:
            payload = item.get("payload", {})
            article_id = payload.get("article_id")
            document_id = payload.get("document_id")

            if not article_id:
                expanded = dict(item)
                expanded_payload = dict(payload)
                expanded_payload["expanded_context_text"] = (
                    payload.get("parent_article_text")
                    or payload.get("matched_clause_text")
                    or ""
                )
                expanded_payload["neighbor_chunks"] = []
                expanded["payload"] = expanded_payload
                expanded_results.append(expanded)
                continue

            must = [
                models.FieldCondition(
                    key="article_id",
                    match=models.MatchValue(value=article_id),
                )
            ]
            if document_id:
                must.append(
                    models.FieldCondition(
                        key="document_id",
                        match=models.MatchValue(value=document_id),
                    )
                )

            neighbor_points, _ = self.client.scroll(
                collection_name=self.collection_name,
                scroll_filter=models.Filter(must=must),
                with_payload=True,
                with_vectors=False,
                limit=64,
            )

            neighbor_payloads = [p.payload for p in neighbor_points if p.payload]
            neighbor_payloads.sort(
                key=lambda x: parse_chunk_order(str(x.get("chunk_id", "")))
            )

            # keep bounded neighbors around full article context
            if len(neighbor_payloads) > max_neighbors:
                neighbor_payloads = neighbor_payloads[:max_neighbors]

            expanded_context_parts = []
            for np in neighbor_payloads:
                ref = np.get("reference_citation") or "N/A"
                unit = np.get("matched_clause_text") or np.get("chunk_text") or ""
                expanded_context_parts.append(f"[{ref}]\n{unit}")

            expanded_context_text = "\n\n".join(expanded_context_parts).strip()
            if not expanded_context_text:
                expanded_context_text = (
                    payload.get("parent_article_text")
                    or payload.get("matched_clause_text")
                    or ""
                )

            expanded = dict(item)
            expanded_payload = dict(payload)
            expanded_payload["neighbor_chunks"] = [
                {
                    "chunk_id": np.get("chunk_id"),
                    "clause_ref": np.get("clause_ref"),
                    "point_ref": np.get("point_ref"),
                }
                for np in neighbor_payloads
            ]
            expanded_payload["expanded_context_text"] = expanded_context_text
            expanded["payload"] = expanded_payload
            expanded_results.append(expanded)

        return expanded_results

    def search(
        self,
        query: str,
        limit: int = 5,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        # Force pipeline params requested by user
        broad_top_k = 12
        rerank_top_l = min(limit, 5)

        t0 = time.perf_counter()
        broad_hits = self.broad_retrieve(
            query=query,
            top_k=broad_top_k,
            is_appendix=is_appendix,
            legal_type=legal_type,
            doc_number=doc_number,
        )
        t1 = time.perf_counter()

        reranked_hits = self.rerank(query=query, candidates=broad_hits, top_l=rerank_top_l)
        t2 = time.perf_counter()

        expanded_hits = self.expand_context(refined_hits=reranked_hits, max_neighbors=8)
        t3 = time.perf_counter()

        for idx, item in enumerate(expanded_hits, start=1):
            item["pipeline_metrics"] = {
                "broad_count": len(broad_hits),
                "rerank_count": len(reranked_hits),
                "final_count": len(expanded_hits),
                "broad_seconds": t1 - t0,
                "rerank_seconds": t2 - t1,
                "expand_seconds": t3 - t2,
                "total_seconds": t3 - t0,
            }
            item["final_rank"] = idx

        return expanded_hits


def print_retrieval_results(results):
    if not results:
        print("Khong tim thay ket qua phu hop.")
        return

    for idx, item in enumerate(results, start=1):
        payload = item["payload"]
        matched_clause = (payload.get("matched_clause_text") or "").replace("\n", " ")
        expanded_ctx = (payload.get("expanded_context_text") or "").replace("\n", " ")
        sectors = payload.get("legal_sectors") or []
        rerank_score = item.get("rerank_score", None)

        print(f"[{idx}] RRF={item.get('rrf_score', 0.0):.4f} | RERANK={rerank_score if rerank_score is not None else 'N/A'}")
        print(f"    Title: {payload.get('title')}")
        print(f"    Citation: {payload.get('reference_citation')}")
        # print(f"    Summary: {payload.get('article_summary', 'N/A')}")
        print(f"    Appendix: {payload.get('is_appendix')} | Sectors: {sectors}")
        print(f"    Matched clause: {matched_clause[:240]}...")
        print(f"    Expanded context: {expanded_ctx[:240]}...")

        pms = item.get("pipeline_metrics", {})
        if pms:
            print(
                "    Pipeline: "
                f"broad={pms.get('broad_count')} -> rerank={pms.get('rerank_count')} -> final={pms.get('final_count')} "
                f"| total={pms.get('total_seconds', 0.0):.4f}s"
            )
        print("-" * 140)


# Init retriever with cross-encoder reranker
import torch
# Init retriever with cross-encoder reranker (Bat FP16 neu co GPU)
reranker = BGEReranker(model_name="BAAI/bge-reranker-v2-m3", use_fp16=torch.cuda.is_available())

retriever = LegalHybridRetriever(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    hybrid_encoder=hybrid_encoder,
    hot_sectors=HOT_SECTORS,
    reranker=reranker,
)


def legal_retrieval(
    query: str,
    limit: int = 5,
    is_appendix: Optional[bool] = None,
    legal_type: Optional[str] = None,
    doc_number: Optional[str] = None,
):
    print(f"TRUY VAN: {query}")
    results = retriever.search(
        query=query,
        limit=limit,
        is_appendix=is_appendix,
        legal_type=legal_type,
        doc_number=doc_number,
    )
    print_retrieval_results(results)
    return results


def benchmark_query_latency(
    retriever_obj: LegalHybridRetriever,
    benchmark_queries: List[str],
    top_k: int = 5,
    warmup: int = 1,
):
    if not benchmark_queries:
        raise ValueError("benchmark_queries khong duoc rong")

    for _ in range(max(0, warmup)):
        _ = retriever_obj.search(query=benchmark_queries[0], limit=top_k)

    latencies = []
    for q in benchmark_queries:
        t0 = time.perf_counter()
        _ = retriever_obj.search(query=q, limit=top_k)
        latencies.append(time.perf_counter() - t0)

    avg_latency = sum(latencies) / len(latencies)
    print("\n=== QUERY BENCHMARK (top_k=5) ===")
    print(f"Queries tested: {len(benchmark_queries)}")
    print(f"Avg latency (s/query): {avg_latency:.4f}")
    print(f"Min latency (s): {min(latencies):.4f}")
    print(f"Max latency (s): {max(latencies):.4f}")

    return {
        "queries": len(benchmark_queries),
        "top_k": top_k,
        "avg_latency_s_per_query": avg_latency,
        "min_latency_s": min(latencies),
        "max_latency_s": max(latencies),
    }


# Bench truy van mau cho top_k=5
BENCHMARK_QUERIES = [
    "Nguyen tac quan ly ngan sach nha nuoc la gi?",
    "Dieu kien xu phat vi pham hanh chinh trong linh vuc thue",
    "Quy dinh ve thu tuc nop ho so thue GTGT",
    "To chuc bo may chinh quyen dia phuong duoc quy dinh the nao?",
    "Muc phat khi nop cham bao cao thue",
]

QUERY_BENCHMARK_STATS = benchmark_query_latency(
    retriever_obj=retriever,
    benchmark_queries=BENCHMARK_QUERIES,
    top_k=5,
    warmup=1,
)
print("QUERY_BENCHMARK_STATS:", QUERY_BENCHMARK_STATS)

Chunks: 100%|██████████| 1/1 [00:00<00:00,  7.69it/s]


ResponseHandlingException: The read operation timed out

# Benchmarking & Evaluation

### PHẦN 1: BỘ CHỈ SỐ ĐO LƯỜNG LÕI (CORE METRICS)

#### 1. Đánh giá tốc độ truy vấn DB (Database Performance)
Với Qdrant hay ChromaDB, đo trung bình và "điểm đuôi" (tail latency) khi hệ thống chịu tải.
* **Latency (Độ trễ):** Đo bằng giây/milliseconds cho mỗi query. Theo dõi **P90** và **P99**.
* **QPS/Throughput (Truy vấn/giây):** Khả năng chịu tải đồng thời.

#### 2. Độ liên quan giữa Câu hỏi & Tài liệu (Retrieval Metrics)
Đánh giá Information Retrieval (IR) với tập Ground Truth (*Câu hỏi* -> *Danh sách ID Điều luật mong đợi*).
* **Recall@K (Độ bao phủ):** Tuyệt đối không được bỏ sót căn cứ pháp lý.
* **Precision@K (Độ chính xác truy xuất):** Hữu ích để đưa vào LLM.
* **MRR (Mean Reciprocal Rank):** Đo lường xem tài liệu đúng xuất hiện ở vị trí thứ mấy.
* **NDCG@K:** Đánh giá chất lượng của mô hình Reranker (bge-reranker).

Dưới đây là kịch bản test thời gian chi tiết (QPS, P90, P99) và báo cáo đánh giá Retrieval (Recall@K, MRR).

In [ ]:
import numpy as np
import time

def evaluate_performance(retriever_obj, queries, top_k=10):
    print("="*60)
    print("ĐÁNH GIÁ TỐC ĐỘ TRUY VẤN DB (DATABASE PERFORMANCE)")
    print("="*60)

    # Warmup
    for q in queries[:2]:
        _ = retriever_obj.search(query=q, limit=top_k)

    latencies = []
    for q in queries:
        t0 = time.perf_counter()
        _ = retriever_obj.search(query=q, limit=top_k)
        latencies.append(time.perf_counter() - t0)

    latencies = np.array(latencies)
    avg_lat = np.mean(latencies)
    p90 = np.percentile(latencies, 90)
    p99 = np.percentile(latencies, 99)
    qps = 1.0 / avg_lat if avg_lat > 0 else 0

    print(f"Tổng số query đã test : {len(queries)}")
    print(f"Thời gian trung bình  : {avg_lat:.4f} s/query")
    print(f"Bách phân vị P90      : {p90:.4f} s")
    print(f"Bách phân vị P99      : {p99:.4f} s")
    print(f"Thông lượng (QPS)     : {qps:.2f} queries/sec")
    assert avg_lat < 1.5, f"WARNING: Latency trung bình > 1.5s ({avg_lat:.4f}s), cần tối ưu lại!"
    print("\n")


# Ground truth mock: Dùng chunk_id hoặc document_number mong muốn
GROUND_TRUTH = {
    "Nguyen tac quan ly ngan sach nha nuoc la gi?": ["83/2015/QH13"],
    "Dieu kien xu phat vi pham hanh chinh trong linh vuc thue": ["125/2020/NĐ-CP"],
    "To chuc bo may chinh quyen dia phuong duoc quy dinh the nao?": ["77/2015/QH13"],
}

def evaluate_retrieval_accuracy(retriever_obj, ground_truth, top_k=10):
    print("="*60)
    print(f"ĐÁNH GIÁ ĐỘ CHÍNH XÁC TRUY XUẤT (RETRIEVAL METRICS @ {top_k})")
    print("="*60)

    recalls = []
    mrrs = []

    for query, expected_docs in ground_truth.items():
        results = retriever_obj.search(query=query, limit=top_k)
        retrieved_docs = [r["payload"].get("document_number", "") for r in results]

        # Calculate Recall
        hits = [doc for doc in expected_docs if any(doc in r for r in retrieved_docs)]
        recall = len(hits) / len(expected_docs) if expected_docs else 0
        recalls.append(recall)

        # Calculate MRR
        rank = 0
        for idx, r_doc in enumerate(retrieved_docs, start=1):
            if any(exp in r_doc for exp in expected_docs):
                rank = idx
                break
        mrrs.append(1.0 / rank if rank > 0 else 0.0)

    avg_recall = np.mean(recalls)
    avg_mrr = np.mean(mrrs)

    print(f"Số lượng truy vấn test (Ground Truth mock) : {len(ground_truth)}")
    print(f"Recall@{top_k} (Độ bao phủ)                     : {avg_recall:.4f}")
    print(f"MRR (Mean Reciprocal Rank)                 : {avg_mrr:.4f}")
    print("\nGhi chú: Với ngành luật, Recall quan trọng hơn Precision (tuyệt đối không bỏ sót).")

# ================= KỊCH BẢN CHẠY DEMO =================
test_queries = [
    "Nguyen tac quan ly ngan sach nha nuoc la gi?",
    "Dieu kien xu phat vi pham hanh chinh trong linh vuc thue",
    "Quy dinh ve thu tuc nop ho so thue GTGT",
    "To chuc bo may chinh quyen dia phuong duoc quy dinh the nao?",
    "Muc phat khi nop cham bao cao thue",
] * 4 # Nhân lên để test độ trễ (20 queries)

evaluate_performance(retriever, test_queries, top_k=10)

# Cần mapping với ID văn bản thật trong collection thì kết quả mới > 0
evaluate_retrieval_accuracy(retriever, GROUND_TRUTH, top_k=10)

In [ ]:
# Cell 11: Lưu và tải CSDL về máy (Kaggle/Colab-ready)
import os
import shutil
from pathlib import Path

print("=== KAGGLE/COLAB CHECK LƯU TRỮ ===")
in_colab = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
in_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"IN_COLAB: {in_colab}")
print(f"IN_KAGGLE: {in_kaggle}")
print(f"QDRANT_ACTIVE_MODE: {os.environ.get('QDRANT_ACTIVE_MODE', 'unknown')}")

if in_kaggle:
    qdrant_path_str = os.environ.get("QDRANT_LOCAL_PATH", "/kaggle/working/local_qdrant_data")
else:
    qdrant_path_str = os.environ.get("QDRANT_LOCAL_PATH", "/content/local_qdrant_data")
    
qdrant_path = Path(qdrant_path_str)

if qdrant_path.exists():
    size_bytes = sum(p.stat().st_size for p in qdrant_path.rglob("*") if p.is_file())
    print(f"Qdrant local data size: {size_bytes / (1024**2):.2f} MB @ {qdrant_path}")

    # Nén tệp để chuẩn bị tải xuống
    zip_path = qdrant_path_str + ".zip"
    print(f"Đang nén dữ liệu từ {qdrant_path_str} vào {zip_path}...")
    shutil.make_archive(qdrant_path_str, 'zip', qdrant_path_str)
    print("Nén hoàn tất!")

    if in_colab:
        print("Đang chuẩn bị tải về máy từ Colab...")
        from google.colab import files
        files.download(zip_path)
        print("Nếu hộp thoại tải về không hiện ra, bạn có thể tải tay từ tab 'Files' bên trái.")
    elif in_kaggle:
        print("Trên Kaggle, bạn có thể tải file ZIP trực tiếp từ Output section (bên phải giao diện cạnh Data).")
else:
    print(f"Qdrant local path chưa tồn tại hoặc rỗng: {qdrant_path}")

print("\nLưu ý mượt trên Kaggle/Colab:")
print("1) Để an toàn, hãy chạy ngắt phần Index hoặc chỉ chạy Cell 1 tới Cell Index rồi chạy trực tiếp Cell này.")
print("2) Sau khi file nén tải xuống, giải nén trên máy của bạn (giữ nguyên tên folder `local_qdrant_data`), và thiết lập `QDRANT_LOCAL_PATH` trong mã nguồn backend của bạn.")
